# Model Alpha — Strong Lensing Simulation Pipeline

### A user's guide to producing and extracting mock lensed images

This notebook walks you through **using** the `new_pipeline_mp` simulation
pipeline end to end. The pipeline takes real galaxy images from an HSC catalogue,
treats them as background sources, gravitationally lenses them under a range of
**dark-matter models**, and renders the result as it would be observed by a chosen
**instrument** (LSST, DES, Euclid, Roman). The output is a set of HDF5 files full of
simulated lensed images plus the physical metadata describing each system.

You do **not** need to understand the physics internals to use it. In day-to-day use
you only ever touch two things:

1. **`configs/default.yaml`** — what to simulate and how much of it.
2. **`run_pipeline_mp.py`** — the single command that launches a run.

Everything downstream (sampling, lens building, ray tracing, noise) is handled for you.

**What this tutorial covers**

- Part 1 — Configuring a run (`default.yaml`)
- Part 2 — Launching a run (`run_pipeline_mp.py`)
- Part 3 — Understanding the output files
- Part 4 — Extracting images and metadata back out for analysis

> The physics ("how the sausage is made") is deliberately out of scope here.

## Prerequisites & setup

Before running anything, make sure you have:

**1. The package installed.** The package lives under `src/model_alpha_pipeline_mp/`
(a "src layout"). The `src/` folder is just a container — it never appears in import
statements, so the package is still imported as `model_alpha_pipeline_mp` (exactly as
`run_pipeline_mp.py` does). For that to resolve, the package must be **installed**, which
is what puts it on Python's import path:

```bash
# from the repo root
pip install -e .
```

With a src layout this step is **required** — you can't just run from the repo root
without installing, because `src/` is not on `sys.path`. Note that
`pip install -r requirements.txt` installs the *dependencies* but **not** the package
itself, so use the editable install above (or, as a fallback, set `PYTHONPATH=src`). This
assumes your `pyproject.toml`/`setup.py` declares `src` as the package root
(e.g. `where = ["src"]`), which is what makes `import model_alpha_pipeline_mp` work.

The core dependencies are `numpy`, `scipy`, `matplotlib`, `tqdm`, `torch`, `nflows`,
`lenstronomy`, `pyHalo`, `colossus`, `h5py`, `pandas`, `pyyaml`, `scikit-learn`, `seaborn`.

**2. The input data and model checkpoints** at the paths named in the config:

| Path (relative to project root) | What it is |
|---|---|
| `input_data/deeplens_update.hdf5` | The HSC source/observational catalogue |
| `checkpoints/trained_Lenspop_flow.pt` | Normalising-flow checkpoint (lens population) |
| `checkpoints/trained_Camels_flow.pt` | Normalising-flow checkpoint (CAMELS) |

You don't pass these on the command line — the run script and config point to them for you.
Just make sure the files exist at those locations.

This tutorial assumes those files are already in place.

---
## Part 1 — The config file: `configs/default.yaml`

A run is fully described by `configs/default.yaml`. This is the main thing you edit.
Here is the default, annotated field by field:

```yaml
run:
  simulations_per_permutation: 3      # how many lenses to generate per (DM type x instrument) combo
  n_workers: 4                        # parallel worker processes
  light_profile: INTERPOL             # INTERPOL (real HSC image as source) or SERSIC (analytic profile)
  instruments:                        # which telescopes to simulate observations for
    - Roman_VIS
    - DES
    - Euclid
    - LSST
  dm_types:                           # which dark-matter models to simulate
    - SIDM
    - Axion
    - CDM
    - WDM
  output_dir: output_data             # top-level folder for results

data:
  hsc_catalog: input_data/deeplens_update.hdf5            # background sources
  lenspop_flow_checkpoint: checkpoints/trained_Lenspop_flow.pt
  camels_flow_checkpoint: checkpoints/trained_Camels_flow.pt

output:
  save_format: hdf5
  save_intermediate: false
```

**The two knobs you'll change most often** are `instruments` and `dm_types` (to pick
*what* gets simulated) and `simulations_per_permutation` (to pick *how many*).

### The "permutation" model — how much will this produce?

The pipeline runs **every combination** of dark-matter type and instrument. With the
default config that is 4 DM types x 4 instruments = **16 permutations**, and each
permutation gets `simulations_per_permutation` lenses:

> 4 x 4 x 3 = **48 simulated lenses total**, written across **16 HDF5 files** (one file
> per permutation).

To do a quick smoke test, trim the lists and the count, e.g. a single `CDM` x `Euclid`
permutation with `simulations_per_permutation: 1`.

### A note on `light_profile`

- **`INTERPOL`** uses a real HSC galaxy image as the source light, interpolated onto the
  lens model. This is the realistic default.
- **`SERSIC`** replaces the source (and deflector) light with analytic Sersic profiles.

One automatic override worth knowing: when the instrument is **`Roman_VIS`**, the pipeline
forces `SERSIC` regardless of the config, because smearing from the HSC image PSF would
wash out the resolution advantage Roman is being simulated for. You don't need to set this
yourself — it happens internally.

---
## Part 2 — Running the pipeline: `run_pipeline_mp.py`

`run_pipeline_mp.py` is the entry point. It does only three things:

1. loads `configs/default.yaml`,
2. unpacks the settings, and
3. calls `simulation_parent(...)`, which orchestrates the whole run.

You normally don't edit it — you edit the config and then run this script. For reference,
its body is essentially:

```python
with open(PROJECT_ROOT/"configs"/"default.yaml", "r") as f:
    config = yaml.safe_load(f)

simulation_parent(
    DM_Types=config["run"]["dm_types"],
    instruments=config["run"]["instruments"],
    sim_number_per_permutation=config["run"]["simulations_per_permutation"],
    light_profile=config["run"]["light_profile"],
    observational_data_path=PROJECT_ROOT/config["data"]["hsc_catalog"],
    output_dir=config["run"]["output_dir"],
    n_workers=config["run"]["n_workers"],
)
```

### Launch it

From the project root, simply run:

In [ ]:
# Run from the project root. (In a notebook the leading ! sends it to the shell.)
# This reads configs/default.yaml and launches the full run.
!python run_pipeline_mp.py

**What to expect while it runs**

- Work is spread across `n_workers` processes (`multiprocessing.Pool`).
- The pipeline iterates permutation by permutation. For each one it prints how many sims
  it is running, then a `Wrote strong_lens_<i> for <DM>/<instrument>` line as each lens
  finishes.
- Individual sims that hit a transient sampling error are silently retried with a fresh
  draw, so the occasional `Trying a new simulation due to error` message is normal.

**Tip:** start small. A full default run is 48 lenses and can take a while. Trim
`instruments`, `dm_types`, and `simulations_per_permutation` for your first run.

### Where the output lands, and resumability

Each run creates a **timestamped run directory**, and writes **one HDF5 file per
permutation** inside it:

```
output_data/
└── model_alpha_[YYYY-MM-DD]/
    ├── model_alpha_CDM_Euclid_[YYYY-MM-DD].h5
    ├── model_alpha_CDM_LSST_[YYYY-MM-DD].h5
    ├── model_alpha_WDM_Euclid_[YYYY-MM-DD].h5
    └── ...  (one file per DM-type x instrument combination)
```

Runs are **resumable**: if you re-run and a file already contains `strong_lens_<i>`,
that simulation is skipped rather than recomputed. So you can safely stop and restart,
or grow a dataset by bumping `simulations_per_permutation` and running again.

---
## Part 3 — Understanding the output files

To extract data you need to know how each HDF5 file is laid out. Every per-permutation
file has the same structure:

```
model_alpha_<DM>_<instrument>_<date>.h5
└── images/
    ├── strong_lens_1/
    │   ├── exposure_1_<band>          # the lensed image  (one per band)
    │   ├── exposure_1_<band>_nss      # same system, no DM substructure
    │   ├── theta_e, z_lens, z_source, snr, ...   # metadata datasets
    │   └── ...
    ├── strong_lens_2/
    └── ...
```

**Image datasets.** For each band the instrument observes, there are two arrays:

- `exposure_<i>_<band>` — the lensed image **with** dark-matter substructure.
- `exposure_<i>_<band>_nss` — the **same** system rendered **without** substructure
  (a smooth-lens counterpart). Pairing the two lets you isolate the substructure signal.

The number of bands depends on the instrument (e.g. Euclid writes a single band; LSST/DES
write three). Don't assume a fixed `g/r/i` layout — read the band list from the file
(see the `bands` dataset below). Each image dataset also carries **attributes**
(`filter`, `pixel_scale`, `fov`, `lens_magnitude`, `source_magnitude`, `units`, ...).

**Metadata datasets.** All the scalar/text metadata follows one convention: each dataset
is a small array of UTF-8 strings where the **last element is a human-readable comment**
and everything before it is the value(s):

```
theta_e        -> ["1.23", "Einstein radius in arcseconds"]
exposure_time  -> ["500.0", "500.0", "500.0", "Exposure time in seconds for all bands"]
bands          -> ["g", "r", "i", "Instrument bands in which image was simulated"]
```

So to read a value you take everything except the final entry. The helpers in Part 4
handle this for you.

### Metadata field reference

Always present per simulation:

| Dataset | Meaning |
|---|---|
| `uid` | Simulation number within its batch |
| `DM_type` | Dark-matter model (`CDM`/`WDM`/`SIDM`/`Axion`) |
| `instrument` | Instrument simulated |
| `bands` | List of bands written for this system |
| `z_lens`, `z_source` | Deflector and source redshifts |
| `d_l`, `d_s`, `d_ls` | Angular diameter distances (Gpc) |
| `theta_e` | Einstein radius (arcsec) |
| `host_mass`, `lens_mass` | log10 host-halo and total-lens mass (M_sun) |
| `num_subhalos` | Number of subhalos rendered |
| `host_slope`, `ellipticity` | EPL slope and ellipticity of the host halo |
| `source_pos` | Source position w.r.t. deflector centre (arcsec) |
| `snr` | Signal-to-noise of the lensed image |
| `exposure_time` | Exposure time per band (s) |
| `light_profile` | `INTERPOL` or `SERSIC` |
| `sigma_sub`, `r_tidal`, `log_mlow`, `log_mhigh` | Substructure sampling parameters |

Some fields appear **only for certain DM types** — e.g. `log_mc` (WDM); `m_axion`,
`flucs_shape`, `flucs_args` (Axion); `subhalo_mass_ranges`, `field_halo_mass_ranges`,
`prob_subhalo`, `prob_field_halo` (SIDM). In `SERSIC` runs you also get `source_sersic`
and `deflector_sersic` shape parameters. Extraction code should treat these as optional.

---
## Part 4 — Extracting the data

Below is a small, self-contained reader built directly on the schema above. It needs only
`h5py`, `numpy`, `pandas`, and `matplotlib`. Point it at one of the output files and it
will give you back images and metadata as ordinary Python objects.

> If your project has a dedicated loader module, prefer that. These helpers are a
> dependency-light fallback that works straight off the on-disk format.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt

In [ ]:
def find_output_file(output_dir, dm_type, instrument, run_dir=None):
    """Locate a per-permutation HDF5 file, resolving the timestamp for you.

    If run_dir is None, the most recent model_alpha_<date>/ directory is used.
    """
    if run_dir is None:
        run_dirs = sorted(glob.glob(os.path.join(output_dir, "model_alpha_*")))
        if not run_dirs:
            raise FileNotFoundError(f"No run directories found under {output_dir!r}")
        run_dir = run_dirs[-1]
    pattern = os.path.join(run_dir, f"model_alpha_{dm_type}_{instrument}_*.h5")
    matches = sorted(glob.glob(pattern))
    if not matches:
        raise FileNotFoundError(f"No file matching {pattern!r}")
    return matches[-1]

In [ ]:
def _to_str(x):
    """HDF5 string datasets come back as bytes or str depending on h5py version."""
    return x.decode() if isinstance(x, (bytes, bytearray)) else str(x)


def read_meta(group, key):
    """Return (values, comment) for a metadata dataset stored as [*values, comment]."""
    arr = [_to_str(v) for v in group[key][()]]
    return arr[:-1], arr[-1]


def read_meta_value(group, key):
    """Convenience: the value(s) only. A lone value is returned unwrapped."""
    values, _comment = read_meta(group, key)
    return values[0] if len(values) == 1 else values


def list_simulations(h5file):
    """Sorted list of simulation indices present in the file."""
    return sorted(int(k.rsplit("_", 1)[-1]) for k in h5file["images"].keys())

In [ ]:
def load_simulation(h5file, i):
    """Load one strong lens: its bands, metadata, and image arrays.

    Returns a dict with keys: 'uid', 'bands', 'metadata',
    'images' (with substructure) and 'images_nss' (without).
    """
    g = h5file[f"images/strong_lens_{i}"]
    bands, _ = read_meta(g, "bands")

    sim = {"uid": i, "bands": bands, "metadata": {}, "images": {}, "images_nss": {}}

    # Every dataset that isn't an image array is metadata.
    for key in g.keys():
        if key.startswith("exposure_"):
            continue
        sim["metadata"][key] = read_meta_value(g, key)

    # Image arrays + their attributes, per band.
    for band in bands:
        dset = g[f"exposure_{i}_{band}"]
        sim["images"][band] = {
            "data": dset[()],
            "attrs": {k: [_to_str(v) for v in dset.attrs[k]] for k in dset.attrs},
        }
        sim["images_nss"][band] = g[f"exposure_{i}_{band}_nss"][()]

    return sim

### Open a file and see what's inside

Set `OUTPUT_DIR`, `DM_TYPE`, and `INSTRUMENT` to match a run you've done.

In [ ]:
OUTPUT_DIR = "output_data"
DM_TYPE = "CDM"
INSTRUMENT = "Euclid"

path = find_output_file(OUTPUT_DIR, DM_TYPE, INSTRUMENT)
print("Reading:", path)

with h5py.File(path, "r") as f:
    sims = list_simulations(f)
    print("Simulations in this file:", sims)

### Inspect a single lens's metadata

In [ ]:
with h5py.File(path, "r") as f:
    sim = load_simulation(f, sims[0])

print(f"strong_lens_{sim['uid']}  |  bands = {sim['bands']}")
for key in ("DM_type", "instrument", "z_lens", "z_source", "theta_e",
            "num_subhalos", "snr", "light_profile"):
    if key in sim["metadata"]:
        print(f"  {key:14s}: {sim['metadata'][key]}")

### Build a metadata table across all simulations

Handy for filtering/selecting systems before you pull the (larger) image arrays.

In [ ]:
def metadata_table(h5file):
    rows = []
    for i in list_simulations(h5file):
        g = h5file[f"images/strong_lens_{i}"]
        row = {"uid": i}
        for key in g.keys():
            if key.startswith("exposure_"):
                continue
            values, _ = read_meta(g, key)
            row[key] = values[0] if len(values) == 1 else ";".join(values)
        rows.append(row)
    return pd.DataFrame(rows).set_index("uid")


with h5py.File(path, "r") as f:
    df = metadata_table(f)

df.head()

### Visualise a lensed image (with vs. without substructure)

Each band gives you the substructure image and its smooth counterpart. Plotting them
side by side is the quickest sanity check that a run produced sensible lenses.

In [ ]:
with h5py.File(path, "r") as f:
    sim = load_simulation(f, sims[0])

bands = sim["bands"]
fig, axes = plt.subplots(2, len(bands), figsize=(4 * len(bands), 8), squeeze=False)

for col, band in enumerate(bands):
    img = sim["images"][band]["data"]
    img_nss = sim["images_nss"][band]

    axes[0, col].imshow(img, origin="lower", cmap="magma")
    axes[0, col].set_title(f"{band} - with substructure")
    axes[1, col].imshow(img_nss, origin="lower", cmap="magma")
    axes[1, col].set_title(f"{band} - no substructure")

    for ax in (axes[0, col], axes[1, col]):
        ax.set_xticks([]); ax.set_yticks([])

fig.suptitle(f"{sim['metadata'].get('DM_type')} / "
             f"{sim['metadata'].get('instrument')} - strong_lens_{sim['uid']}")
plt.tight_layout()
plt.show()

---
## Recap

You now have the full loop:

1. **Configure** a run by editing `configs/default.yaml` — pick your `instruments`,
   `dm_types`, and `simulations_per_permutation`.
2. **Run** it with `python run_pipeline_mp.py`. Output lands in a timestamped folder,
   one HDF5 file per DM-type x instrument permutation, and runs are resumable.
3. **Extract** images and metadata with `h5py`, using the `[value..., comment]` convention
   and the `exposure_<i>_<band>` / `_nss` image pairs.

From here you can select systems with the metadata table and feed the image arrays into
whatever analysis or training pipeline you like.

> If you have a dedicated output-loader module, drop it in and replace the Part 4 helpers
> with its API — the rest of the tutorial stays the same.